In [1]:
# peft = LoRA adapters; bitsandbytes = 4-bit quantisation (the "Q" in QLoRA).
# langchain-openai, python-dotenv and chromadb are needed only so this notebook can
# import the REPO's own prompt, metrics and retrieval code, instead of copying it.
!pip install -q -U peft bitsandbytes
!pip install -q langchain-openai python-dotenv chromadb

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 835.4/835.4 kB 28.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 46.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 145.6/145.6 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 59.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 15.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.5/7.5 MB 102.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 572.8/572.8 kB 29.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 76.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.1/140.1 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.3/60.3 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20

In [2]:
# Clone the exact code that produced the local baselines. /tmp, not /kaggle/working,
# so the repo doesn't clutter the saved outputs.
!git clone -q https://github.com/akshatyuvan/autonomous-preprocessing-agent.git /tmp/repo
!git -C /tmp/repo log --oneline -1

5615a5f (HEAD -> main, origin/main, origin/HEAD) Docker: let the non-root user write under /app so pipeline snapshots can be saved


In [3]:
import json, math, os, random, re, sys, time
from contextlib import nullcontext
from pathlib import Path

os.chdir("/tmp/repo")          # the repo's code uses paths like evaluation/data/...
sys.path.insert(0, "/tmp/repo")

# True  = 3 training steps + 8 test rows per condition: proves everything runs (~20 min).
# False = the real run: full training, all 100 test rows, all 4 conditions.
SMOKE = False

MODEL_ID = "meta-llama/Llama-3.2-3B-Instruct"
OUT = Path("/kaggle/working")
(OUT / "results").mkdir(parents=True, exist_ok=True)
SEED = 42

# LoRA: train small low-rank matrices next to the frozen weights instead of all 3B weights.
# r = rank of those matrices (capacity); alpha/r scales their contribution (here 2x).
LORA_R, LORA_ALPHA, LORA_DROPOUT = 16, 32, 0.05
# Effective batch = BATCH * ACCUM = 16 examples per optimiser step.
# 300 examples -> 19 steps/epoch -> 57 steps over 3 epochs (fixed in advance, not tuned).
EPOCHS, LR, BATCH, ACCUM, MAX_LEN = 3, 2e-4, 2, 8, 2048
MAX_NEW_TOKENS, GEN_BATCH = 200, 8

In [4]:
# The token comes from Kaggle Secrets, so it never appears in the notebook or its outputs.
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient
login(token=UserSecretsClient().get_secret("HF_TOKEN"))

In [5]:
import numpy as np
import torch

# The REPO's own code: same prompt builder, same verdict parser, same metrics.
from agents.critic import DecisionJudgment, build_critic_messages
from evaluation.compare import mcnemar
from evaluation.metrics import critic_metrics
from evaluation.run_critic_eval import _rule, load_jsonl
from vector_store.retrieval import HybridRetriever, default_embedder, load_cases

train = load_jsonl(Path("evaluation/data/critic_train.jsonl"))
test = load_jsonl(Path("evaluation/data/critic_test.jsonl"))
assert len(train) == 300 and len(test) == 100
assert not {e["id"] for e in train} & {e["id"] for e in test}  # no leakage
if SMOKE:
    test = test[:8]

# Locally, Ollama forces JSON through tool calling. Plain generation has no tool
# calling, so ALL FOUR conditions get this same instruction: fair within the table.
# "reason" comes BEFORE "verdict" so the model states the numbers before deciding.
FORMAT = ('\n\nRespond with ONLY a JSON object and nothing else: '
          '{"reason": "<one sentence citing the rule number and the numbers>", '
          '"verdict": "accept" or "reject"}')


def chat(evidence, patterns=None):
    (_, system), (_, human) = build_critic_messages(evidence, patterns)
    return [{"role": "system", "content": system}, {"role": "user", "content": human + FORMAT}]


def target(example):
    # Training answer = the policy rule that labelled the example, then the label.
    return json.dumps({"reason": example["label_rule"], "verdict": example["label"]})


def parse(text):
    """(verdict, reason), or (None, "") when the output is unusable = malformed."""
    m = re.search(r'"verdict"\s*:\s*"([^"]*)"', text)
    if not m:
        return None, ""
    try:
        verdict = DecisionJudgment(verdict=m.group(1), reason="").verdict  # same normaliser as live
    except Exception:
        return None, ""
    r = re.search(r'"reason"\s*:\s*"([^"]*)"', text)
    return verdict, (r.group(1) if r else "")


print(len(train), "train /", len(test), "test", "(SMOKE)" if SMOKE else "")

300 train / 100 test 


In [6]:
from transformers import (AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig,
                          get_cosine_schedule_with_warmup)

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

tok = AutoTokenizer.from_pretrained(MODEL_ID)
tok.pad_token = tok.eos_token  # Llama has no pad token; padding is masked out anyway

# QLoRA's base model: 4-bit NF4 weights (~2 GB instead of ~6 GB), double quantisation,
# fp16 compute (a T4 has no bf16).
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                         bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.float16)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, quantization_config=bnb,
                                             device_map={"": 0}, torch_dtype=torch.float16)
print(torch.cuda.get_device_name(0), f"{torch.cuda.memory_allocated() / 1e9:.1f} GB used")

config.json:   0%|          | 0.00/878 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/54.5k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

Tesla T4 2.2 GB used


In [7]:
def encode(example):
    prompt = tok.apply_chat_template(chat(example["evidence"]), tokenize=False,
                                     add_generation_prompt=True)
    p = tok(prompt, add_special_tokens=False).input_ids
    r = tok(target(example) + tok.eos_token, add_special_tokens=False).input_ids
    # labels = -100 on the prompt: the loss is computed ONLY on the answer, so the
    # model learns to answer, not to reproduce the prompt.
    return {"input_ids": p + r, "labels": [-100] * len(p) + r}


encoded = [encode(e) for e in train]
lengths = [len(e["input_ids"]) for e in encoded]
print(f"tokens per example: max {max(lengths)}, mean {sum(lengths) / len(lengths):.0f}")
assert max(lengths) <= MAX_LEN, "an example is longer than MAX_LEN"

tokens per example: max 814, mean 708


In [8]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# Gradient checkpointing: recompute activations in the backward pass instead of storing
# them all -- trades compute for memory so this fits on a 16 GB T4.
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
model = get_peft_model(model, LoraConfig(
    r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT, bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]))
model.print_trainable_parameters()  # a tiny fraction of 3B: the whole point of LoRA


def collate(batch):
    width = max(len(b["input_ids"]) for b in batch)
    pad = lambda seq, value: seq + [value] * (width - len(seq))
    return {
        "input_ids": torch.tensor([pad(b["input_ids"], tok.pad_token_id) for b in batch], device="cuda"),
        "labels": torch.tensor([pad(b["labels"], -100) for b in batch], device="cuda"),
        "attention_mask": torch.tensor([pad([1] * len(b["input_ids"]), 0) for b in batch], device="cuda"),
    }


steps_per_epoch = math.ceil(len(encoded) / (BATCH * ACCUM))
total_steps = 3 if SMOKE else EPOCHS * steps_per_epoch
params = [p for p in model.parameters() if p.requires_grad]  # only the LoRA matrices
opt = torch.optim.AdamW(params, lr=LR, weight_decay=0.0)
sched = get_cosine_schedule_with_warmup(opt, num_warmup_steps=max(1, int(0.1 * total_steps)),
                                        num_training_steps=total_steps)
scaler = torch.amp.GradScaler("cuda")  # fp16 needs loss scaling, or small gradients underflow to 0

model.train()
model.config.use_cache = False  # the generation cache is useless (and wasteful) in training
order_rng = random.Random(SEED)
log, step, running, micro, started, finished = [], 0, 0.0, 0, time.time(), False

for epoch in range(EPOCHS):
    order = list(range(len(encoded)))
    order_rng.shuffle(order)
    batches = [order[i:i + BATCH] for i in range(0, len(order), BATCH)]
    for i, idx in enumerate(batches):
        with torch.autocast("cuda", dtype=torch.float16):
            loss = model(**collate([encoded[j] for j in idx])).loss / ACCUM
        scaler.scale(loss).backward()
        running += loss.item() * ACCUM
        micro += 1
        if (i + 1) % ACCUM == 0 or i + 1 == len(batches):  # one optimiser step per ACCUM batches
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(params, 1.0)
            scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True); sched.step()
            step += 1
            log.append({"step": step, "epoch": epoch + 1, "loss": round(running / micro, 4),
                        "lr": sched.get_last_lr()[0], "elapsed_s": round(time.time() - started)})
            print(log[-1])
            running, micro = 0.0, 0
            if step >= total_steps:
                finished = True
                break
    if finished:
        break

model.save_pretrained(OUT / "critic_lora_adapter")  # only the adapter (MBs), not the 3B base
(OUT / "training_log.json").write_text(json.dumps({
    "smoke": SMOKE, "model": MODEL_ID, "steps": step, "train_examples": len(train),
    "lora": {"r": LORA_R, "alpha": LORA_ALPHA, "dropout": LORA_DROPOUT},
    "epochs": EPOCHS, "lr": LR, "effective_batch": BATCH * ACCUM, "log": log,
    "train_seconds": round(time.time() - started)}, indent=2))
print("adapter saved")

trainable params: 24,313,856 || all params: 3,237,063,680 || trainable%: 0.7511


/usr/local/lib/python3.13/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


{'step': 1, 'epoch': 1, 'loss': 1.3291, 'lr': 4e-05, 'elapsed_s': 22}
{'step': 2, 'epoch': 1, 'loss': 1.2, 'lr': 8e-05, 'elapsed_s': 42}
{'step': 3, 'epoch': 1, 'loss': 0.9728, 'lr': 0.00012, 'elapsed_s': 63}
{'step': 4, 'epoch': 1, 'loss': 0.9663, 'lr': 0.00016, 'elapsed_s': 84}
{'step': 5, 'epoch': 1, 'loss': 0.7045, 'lr': 0.0002, 'elapsed_s': 105}
{'step': 6, 'epoch': 1, 'loss': 0.5624, 'lr': 0.00019981755542233177, 'elapsed_s': 127}
{'step': 7, 'epoch': 1, 'loss': 0.419, 'lr': 0.0001992708874098054, 'elapsed_s': 149}
{'step': 8, 'epoch': 1, 'loss': 0.3872, 'lr': 0.00019836199069471437, 'elapsed_s': 171}
{'step': 9, 'epoch': 1, 'loss': 0.1615, 'lr': 0.0001970941817426052, 'elapsed_s': 194}
{'step': 10, 'epoch': 1, 'loss': 0.138, 'lr': 0.00019547208665085457, 'elapsed_s': 217}
{'step': 11, 'epoch': 1, 'loss': 0.112, 'lr': 0.0001935016242685415, 'elapsed_s': 240}
{'step': 12, 'epoch': 1, 'loss': 0.1598, 'lr': 0.00019118998459920902, 'elapsed_s': 265}
{'step': 13, 'epoch': 1, 'loss': 0

In [9]:
@torch.no_grad()
def judge_all(examples, retriever=None):
    model.eval()
    model.config.use_cache = True
    tok.padding_side = "left"  # generation pads on the LEFT so every prompt ends at the same position
    rows = []
    for start in range(0, len(examples), GEN_BATCH):
        chunk = examples[start:start + GEN_BATCH]
        prompts = [tok.apply_chat_template(
                       chat(e["evidence"], retriever.retrieve(e["evidence"]) if retriever else None),
                       tokenize=False, add_generation_prompt=True) for e in chunk]
        enc = tok(prompts, return_tensors="pt", padding=True, add_special_tokens=False).to("cuda")
        t0 = time.time()
        out = model.generate(**enc, max_new_tokens=MAX_NEW_TOKENS, do_sample=False,
                             pad_token_id=tok.pad_token_id)  # greedy = deterministic
        secs = round((time.time() - t0) / len(chunk), 2)
        texts = tok.batch_decode(out[:, enc["input_ids"].shape[1]:], skip_special_tokens=True)
        for e, text in zip(chunk, texts):
            prediction, reason = parse(text)
            # Same row format as the local runs, so metrics/compare/MLflow work unchanged.
            rows.append({"id": e["id"], "scenario": e["scenario"], "strategy": e["strategy"],
                         "borderline": e["borderline"], "rule": _rule(e), "label": e["label"],
                         "prediction": prediction, "reason": reason,
                         "error": None if prediction else "malformed", "seconds": secs,
                         "raw_output": text[:500]})
        print(f"  {len(rows)}/{len(examples)}")
    return rows

In [10]:
import bitsandbytes, peft, transformers

retriever = HybridRetriever(load_cases(), default_embedder())  # TRAIN split only
commit = os.popen("git -C /tmp/repo rev-parse --short HEAD").read().strip()
conditions = [  # (name, adapter on?, retrieval on?)
    ("kaggle_base", False, False),
    ("kaggle_base_rag", False, True),
    ("kaggle_ft", True, False),
    ("kaggle_ft_rag", True, True),
]
saved = {}
for name, use_adapter, use_rag in conditions:
    print(name)
    # disable_adapter(): the SAME loaded model with the LoRA switched off = the base model.
    with (nullcontext() if use_adapter else model.disable_adapter()):
        rows = judge_all(test, retriever if use_rag else None)
    saved[name] = rows
    (OUT / "results" / f"{name}.json").write_text(json.dumps({
        "condition": name, "provider": "transformers-kaggle", "model": f"{MODEL_ID} (4-bit NF4)",
        "adapter": use_adapter, "retrieval": use_rag, "smoke": SMOKE,
        "test_file": "evaluation/data/critic_test.jsonl", "repo_commit": commit,
        "gpu": torch.cuda.get_device_name(0),
        "versions": {"torch": torch.__version__, "transformers": transformers.__version__,
                     "peft": peft.__version__, "bitsandbytes": bitsandbytes.__version__},
        "metrics": critic_metrics(rows), "predictions": rows}, indent=2))
    print(json.dumps({k: v for k, v in critic_metrics(rows).items()
                      if k in ("accuracy", "reject_precision", "reject_recall", "malformed_rate")}))

/root/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz: 100%|██████████| 79.3M/79.3M [00:00<00:00, 105MiB/s]


kaggle_base


[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


  8/100
  16/100
  24/100
  32/100
  40/100
  48/100
  56/100
  64/100
  72/100
  80/100
  88/100
  96/100
  100/100
{"malformed_rate": 0.01, "accuracy": 0.58, "reject_precision": 0.5, "reject_recall": 0.286}
kaggle_base_rag
  8/100
  16/100
  24/100
  32/100
  40/100
  48/100
  56/100
  64/100
  72/100
  80/100
  88/100
  96/100
  100/100
{"malformed_rate": 0.0, "accuracy": 0.72, "reject_precision": 0.889, "reject_recall": 0.381}
kaggle_ft
  8/100
  16/100
  24/100
  32/100
  40/100
  48/100
  56/100
  64/100
  72/100
  80/100
  88/100
  96/100
  100/100
{"malformed_rate": 0.0, "accuracy": 1.0, "reject_precision": 1.0, "reject_recall": 1.0}
kaggle_ft_rag
  8/100
  16/100
  24/100
  32/100
  40/100
  48/100
  56/100
  64/100
  72/100
  80/100
  88/100
  96/100
  100/100
{"malformed_rate": 0.0, "accuracy": 0.86, "reject_precision": 0.889, "reject_recall": 0.762}


In [11]:
cols = ["accuracy", "reject_precision", "reject_recall", "malformed_rate",
        "accuracy_lookup_rules", "accuracy_threshold_rules"]
lines = ["| Condition | " + " | ".join(cols) + " |", "|" + "---|" * (len(cols) + 1)]
for name, rows in saved.items():
    m = critic_metrics(rows)
    lines.append(f"| {name} | " + " | ".join(str(m.get(c)) for c in cols) + " |")
table = "\n".join(lines)
print(table)

pairs = [("kaggle_base", "kaggle_base_rag"), ("kaggle_base", "kaggle_ft"),
         ("kaggle_ft", "kaggle_ft_rag"), ("kaggle_base", "kaggle_ft_rag")]
comparisons = [{"a": a, "b": b, **mcnemar(saved[a], saved[b])} for a, b in pairs]
for c in comparisons:
    print(c)

(OUT / "results_table.md").write_text(table + "\n")
(OUT / "comparisons.json").write_text(json.dumps(comparisons, indent=2))

| Condition | accuracy | reject_precision | reject_recall | malformed_rate | accuracy_lookup_rules | accuracy_threshold_rules |
|---|---|---|---|---|---|---|
| kaggle_base | 0.58 | 0.5 | 0.286 | 0.01 | 0.69 | 0.535 |
| kaggle_base_rag | 0.72 | 0.889 | 0.381 | 0.0 | 0.862 | 0.662 |
| kaggle_ft | 1.0 | 1.0 | 1.0 | 0.0 | 1.0 | 1.0 |
| kaggle_ft_rag | 0.86 | 0.889 | 0.762 | 0.0 | 0.966 | 0.817 |
{'a': 'kaggle_base', 'b': 'kaggle_base_rag', 'n_examples': 100, 'accuracy_a': 0.58, 'accuracy_b': 0.72, 'a_only_correct': 7, 'b_only_correct': 21, 'p_value': 0.0125}
{'a': 'kaggle_base', 'b': 'kaggle_ft', 'n_examples': 100, 'accuracy_a': 0.58, 'accuracy_b': 1.0, 'a_only_correct': 0, 'b_only_correct': 42, 'p_value': 0.0}
{'a': 'kaggle_ft', 'b': 'kaggle_ft_rag', 'n_examples': 100, 'accuracy_a': 1.0, 'accuracy_b': 0.86, 'a_only_correct': 14, 'b_only_correct': 0, 'p_value': 0.0001}
{'a': 'kaggle_base', 'b': 'kaggle_ft_rag', 'n_examples': 100, 'accuracy_a': 0.58, 'accuracy_b': 0.86, 'a_only_correct': 3,

802